<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/cst2312_h11/blob/main/CST2312_Module08_Web_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CST2312 Module 8  Web APIs**

*This notebook is available at -- [bit.ly/cst2312WebAPIs](https://bit.ly/cst2312WebAPIs)*

> *includes exercises to be completed as a class assignment*


### Required Reading

**PY4E Chapter 13 — Using Web Services** (required)  
[https://www.py4e.com/html3/13-web](https://www.py4e.com/html3/13-web)
  
### Supplementary Reading

**FOPP Chapter 24 — Internet APIs** (supplemental; free Runestone account required)  
[https://runestone.academy/ns/books/published/fopp/InternetAPIs/toctree.html](https://runestone.academy/ns/books/published/fopp/InternetAPIs/toctree.html)

---

## Learning Objectives

By the end of this module you will be able to:

- Explain what a REST API is and how it differs from a local function call
- Use Python's `requests` library to send HTTP GET requests and inspect the response
- Parse a JSON response into a Python dictionary and extract specific fields
- Distinguish between APIs that require an API key and those that do not
- Store API keys securely using the Colab Secrets tab
- Pass query parameters to an API using a Python dictionary and `requests.get()`
- Chain two APIs together so that the output of one becomes the input of the next
- Create a simple interactive map with the `folium` library using coordinates from an API response

## Connections to Prior Material

This module draws directly on several topics covered earlier in the course:

| Prior topic | How it appears here |
|---|---|
| **Dictionaries** (Module 11) | JSON responses parse into Python dictionaries; every field access uses `data["key"]` |
| **Functions** (earlier modules) | API calls are wrapped in reusable functions that accept parameters and return data |
| **Exception handling** (Module 09) | Network requests can fail; `try/except` blocks catch HTTP errors and timeouts |
| **f-strings** | Used to build URLs with variable components |

---

## Setup

Run this cell once before executing any other code in the notebook.

In [ ]:
import requests
import json
from google.colab import userdata   # Colab Secrets — used for API keys

---

## What Is an API?

A **Web API** (Application Programming Interface) is a service exposed over the internet
that accepts structured requests and returns structured data — typically JSON.
Calling a web API is conceptually identical to calling a Python function:

- The **URL** identifies the function (endpoint)
- **Query parameters** in the URL are the function's arguments
- The **JSON response** is the return value

The following URL is a complete API call you can paste into any browser:

[https://restcountries.com/v3.1/name/canada](https://restcountries.com/v3.1/name/canada)

Try it with different country names. Each response is a JSON document —
a structure that maps directly onto the Python dictionaries and lists you already know.

### JSON and Python Dictionaries

**JSON** (JavaScript Object Notation) is the standard format for data exchange between web services.
A JSON object is structurally identical to a Python dictionary:
key-value pairs, nested objects, and arrays that correspond to Python lists.

```json
{
  "name": {"common": "Canada"},
  "capital": ["Ottawa"],
  "population": 38005238,
  "region": "Americas"
}
```

When you call `response.json()`, the `requests` library parses this text and returns a Python
dictionary. Every field is then accessible with the familiar `data["key"]` syntax.

### Service-Oriented Architecture (SOA)

Most non-trivial applications do not do everything themselves.
They consume services provided by other applications:

- An e-commerce site calls a payment processor API to charge a credit card
- A travel site calls hotel reservation APIs to compare availability
- A financial dashboard calls a market data API to retrieve stock prices

Each service publishes an API — a documented contract specifying how requests must be formed
and what responses will look like. This design pattern is called **Service-Oriented Architecture (SOA)**.

REST (Representational State Transfer) is the dominant architectural style for web APIs today.

| Property | Meaning |
|---|---|
| **Stateless** | Each request contains all information needed; the server keeps no session state |
| **Resource-based** | Data is addressed by URL (e.g., `/v3.1/name/canada`) |
| **HTTP methods** | `GET` retrieves data; `POST` submits; `PUT` updates; `DELETE` removes |
| **Standard formats** | Responses in JSON (most common) or XML |

---

## Web APIs with Python — No API Key Required

Some APIs are open and require no authentication. They are useful for learning
because setup is immediate. We will use two:

- **`uselessfacts.jsph.pl`** — returns a random fact as JSON
- **`restcountries.com`** — returns geographic and demographic data for any country

### The `requests` Library and the Response Object

Two attributes of a `Response` object are most important:

| Attribute / Method | Description |
|---|---|
| `response.status_code` | Integer HTTP status (200 = success; 404 = not found; 5xx = server error) |
| `response.text` | Raw response body as a string |
| `response.json()` | Parses the response body and returns a Python dict or list |



---



### Example 1: Random Fact (uselessfacts.jsph.pl)

In [ ]:
# A simple GET request — no API key required
url = "https://uselessfacts.jsph.pl/api/v2/facts/random"

response = requests.get(url)

print("Status code:", response.status_code)
print()
print("Raw response text:")
print(response.text)

The raw response is a JSON string.
Calling `.json()` converts it into a Python dictionary so we can access individual fields.

In [ ]:
data = response.json()
print(type(data))   # confirms it is a dict
print()
print("Fact:  ", data["text"])
print("Source:", data["source"])

We can iterate through the keys in the resultant dictionarry to explore the data available.

In [ ]:
print(data.keys())

In [ ]:
print("The response information - ")
for key in data:
    print(f"\t{key}:  {data[key]}")

### Example 2: `ipify` - The Current Session's IP Address

In [ ]:
# obtain the public IP address of the current Colab session
my_ip = requests.get("https://api.ipify.org").text
print(f"This session is connected to the internet as {my_ip}.")

---

## API Keys and Secure Storage

Most APIs that provide valuable or high-volume data require an **API key** —
a credential that identifies who is making each request. Keys enable providers to:

- Enforce **rate limits** (e.g., 1,000 requests/day on a free tier)
- Track usage for **billing**
- Revoke access if a key is misused

### Why API Keys Must Not Appear in Code

If a key is pasted directly into a notebook that is shared on Brightspace, pushed to GitHub,
or opened on another machine, anyone who reads the file can consume — or exhaust — your quota.

### Storing Keys in Colab Secrets

Click the **key icon** in the left sidebar to open the Secrets tab.
Add a name-value pair for each API key you use. Access it in code with:

```python
from google.colab import userdata
api_key = userdata.get("myAPIkeyName")
```

The string passed to `userdata.get()` must match the name in the Secrets tab exactly.

| Storage method | Persists after session? | Portable outside Google? |
|---|---|---|
| Colab Secrets tab | Yes | No |
| Google Secret Manager | Yes | No (requires GCP auth) |
| Encrypted local drive ([VeraCrypt](https://www.veracrypt.fr)) | Yes | Yes |

For this course, **Colab Secrets** is the recommended approach.

In [ ]:
api_key = userdata.get("myAPIkeyName")
print(f"The API key for 'myAPIkeyName` in Secrets is: '{api_key}'.")

## Obtaining a REST Countries API Key

The REST Countries API requires an API key for every request. Each request sends the key in an `Authorization` header. The free plan allows 1,000 requests per month, which is sufficient for the work in this notebook.

### Step 1: Create an Account

1. Go to [restcountries.com](https://restcountries.com) and select **Get an API key** in the page header.
2. Complete the sign-up form with an email address and a password.
3. Select the **Free** plan. No credit card is required for the free plan.

A small number of new accounts are placed in a manual review queue, which can take up to approximately 12 hours. Create the account before the class session in which the key is needed.

### Step 2: Copy the API Key

1. After signing in, open the **API Keys** page at [restcountries.com/api-keys](https://restcountries.com/api-keys).
2. Copy the key. A key begins with the prefix `rc_live_`.

Treat the key as a password. Anyone who has the key can send requests that count against the account's monthly limit. If a key is exposed, select **Revoke** on the API Keys page and create a replacement. Revocation takes effect immediately.

### Step 3: Store the Key in Colab Secrets

The key must never be typed into a code cell. A notebook shared with another person, submitted for grading, or saved to GitHub carries every value written in its cells. Colab Secrets stores the key outside the notebook.

1. In the left sidebar of Colab, select the key icon (**Secrets**).
2. Select **Add new secret**.
3. In the **Name** field, enter `restcountriesAPIkey`. The name must match exactly, including capitalization, because the code below retrieves the key by this name.
4. In the **Value** field, paste the API key.
5. Turn on the **Notebook access** toggle for this notebook.

### Step 4: Retrieve the Key in Code

The cell below reads the key from Colab Secrets into the variable `api_key`. Later cells send it in the request header as `"Authorization": f"Bearer {api_key}"`.

```python
from google.colab import userdata

api_key = userdata.get("restcountriesAPIkey")
```

If this cell raises a `SecretNotFoundError`, check the spelling of the secret name. If it raises a `NotebookAccessError`, turn on the **Notebook access** toggle from Step 3.

<font color=blue><i>note: Be sure that the `Notebook Access` button is switched on in your Colab Secrets area when attempting to use a secret in a notebook.<i></font>

### Example: Country Data (restcountries.com)

`restcountries.com` returns a list containing one dictionary per matching country.
The response includes population, capital, region, languages, currency, flag, coordinates, and more.

In [ ]:
url = "https://api.restcountries.com/countries/v5?q=canada"
# url = "https://api.restcountries.com/countries/v5?q=france"
# url = "https://api.restcountries.com/countries/v5?q=LaLaLand"
###
# Try it by prompting for a country..
# q_country = input("What country would you like? ").strip().lower()
# url = f"https://api.restcountries.com/countries/v5?q={q_country}"
# ###

api_key = userdata.get("restcountriesAPIkey")

try:
    response = requests.get(
        url,
        headers={"Authorization": f"Bearer {api_key}"}
    )

    if response.status_code == 200:
        data = response.json()                    # dict with keys "data" and possibly others
        country = data["data"]["objects"][0]      # first matching country dict
        print("Country:   ", country["names"]["common"])
        print("Capital:   ", country["capitals"][0]["name"])
        print("Population:", country["population"])
        print("Region:    ", country["region"])
    else:
        print(f"Request failed with status code {response.status_code}")
except:
    print(f"Are you sure? Something is wrong with '{url}'.")

Notice the nested access. The expression `country["names"]["common"]` reaches into the `names` dictionary inside the country dictionary. The expression `country["capitals"][0]["name"]` reaches into the first dictionary in the `capitals` list. Both expressions follow the same indexing rules that apply to any Python dictionary or list. The country dictionary itself sits deeper in the response, at `data["data"]["objects"][0]`.

The function below wraps this pattern so that it can be called for any country name. It also illustrates how `try`/`except` handles network or data errors (a pattern introduced in an earlier module).

In [ ]:
def get_country_info(country_name):
    """Return a summary dict of geographic and demographic data for a named country."""
    url = "https://api.restcountries.com/countries/v5"
    api_key = userdata.get("restcountriesAPIkey")
    try:
        response = requests.get(
            url,
            params={"q": country_name},
            headers={"Authorization": f"Bearer {api_key}"}
        )
        if response.status_code == 200:
            matches = response.json()["data"]["objects"]
            if not matches:
                return {"error": f"Country not found: '{country_name}'"}
            country = matches[0]
            capitals = country.get("capitals", [])
            return {
                "name":       country["names"]["common"],
                "capital":    capitals[0]["name"] if capitals else "",
                "population": country["population"],
                "region":     country["region"],
                "languages":  [language["name"] for language in country.get("languages", [])],
                "flag":       country.get("flag", {}).get("emoji", "")
            }
        elif response.status_code == 404:
            return {"error": f"Country not found: '{country_name}'"}
        else:
            return {"error": f"HTTP {response.status_code}"}
    except Exception as e:
        return {"error": str(e)}

In [ ]:
country_name = input("Enter a country name: ")
info = get_country_info(country_name)
for key, value in info.items():
    print(f"{key:>12}: {value}")

---

## API Calls with Query Parameters

The `get_country_info()` function above sent its search term to the REST Countries API through the `params` argument of `requests.get()`:

```python
response = requests.get(
    url,
    params={"q": country_name},
    headers={"Authorization": f"Bearer {api_key}"}
)
```

The dictionary passed to `params` supplies the **query parameters** of the request. A query parameter is a name and value pair that tells the API which data to return. The `requests` library adds the parameters to the end of the URL as a **query string**. The query string begins with `?`, and each `name=value` pair after the first is joined with `&`:

```python
params = {"key1": "value1", "key2": "value2"}
response = requests.get(base_url, params=params)
# → base_url?key1=value1&key2=value2
```

For the REST Countries example, `params={"q": "canada"}` produces the request URL `https://api.restcountries.com/countries/v5?q=canada`. The final URL of any request is available in `response.url`.

Passing a dictionary to `params` has three advantages over typing the query string into the URL:

* **Encoding is automatic.** Spaces and special characters are not allowed in a URL, so `requests` converts them to a safe form. For example, `{"q": "South Africa"}` becomes `?q=South+Africa`.
* **Values are converted to text.** A number such as `{"limit": 5}` is sent as `limit=5` without a separate `str()` call.
* **Parameters can be changed in one place.** Adding, removing, or changing a parameter means editing one dictionary entry rather than rewriting the URL string.

Each API defines its own parameter names and the values it accepts. The names `q`, `limit`, and `offset`, for example, have meaning to the REST Countries API because its documentation defines them. The `meta` section of the REST Countries response shows `limit` and `offset` values for the current request. Before calling a new API, consult its documentation for the parameters it supports.

Query parameters and headers serve different purposes. Query parameters describe the data being requested, and they appear in the URL. Headers carry information about the request itself, such as the API key in the `Authorization` header. Because a URL can be recorded in server logs and browser history, an API key is safer in a header than in a query parameter whenever the API accepts it there.


## Example: ipstack — IP Geolocation API

[ipstack.com](https://ipstack.com) returns the geographic location of an IP address: country, region, city, latitude, longitude, ISP, and time zone. The ipstack API requires an API key for every request. The free plan allows 100 requests per month. Every request counts against that limit, including requests that return an error, so run each ipstack cell only as often as needed.

### Step 1: Create an Account

1. Go to [ipstack.com](https://ipstack.com) and select **Get Free API Key**.
2. Complete the sign-up form with an email address and a password.
3. Select the **Free** plan. No credit card is required for the free plan.

### Step 2: Copy the API Key

1. After signing in, open the [ipstack dashboard](https://ipstack.com/dashboard).
2. Copy the value labeled as the API access key.

Treat the key as a password. Anyone who has the key can send requests that count against the account's monthly limit. The dashboard also reports the number of requests used in the current month.

### Step 3: Store the Key in Colab Secrets

Follow the same procedure used for the REST Countries key:

1. In the left sidebar of Colab, select the key icon (**Secrets**).
2. Select **Add new secret**.
3. In the **Name** field, enter `ipstackAPIkey`. The name must match exactly, including capitalization.
4. In the **Value** field, paste the API key.
5. Turn on the **Notebook access** toggle for this notebook.

### Step 4: Retrieve the Key in Code

```python
from google.colab import userdata

ipstack_key = userdata.get("ipstackAPIkey")
```

### How ipstack Receives the Key

REST Countries receives its key in an `Authorization` header. ipstack receives its key as a query parameter named `access_key` in the URL:

<font color=red><b><i>Note: ipstack.com will limit (throttle) you to 100 API calls per month.</i></b></font>

In [ ]:
#@title Old way to use ipstack.com (for API keys created long ago)

api_key = userdata.get("OLDipstackAPIkey")
# 'check' resolves the IP of the current session
url = f"http://api.ipstack.com/check?access_key={api_key}"

response = requests.get(url)
print("Status code:", response.status_code)

In [ ]:
# NEW WAY TO USE IPSTACK.COM, FOR USE WITH CURRENT API KEYS
# see snippet sample -- https://app.apilayer.com/playground/ipstack

import requests
api_key = userdata.get("ipstackAPIkey")

url = "https://api.apilayer.net/ipstack/134.201.250.155"
params = {
    "access_key": f"{api_key}"
}

response = requests.get(url, params=params)
print(response.json())

In [ ]:
data = response.json()
print(data)

In [ ]:
print("IP address:", data["ip"])
print("City:      ", data["city"])
print("Country:   ", data["country_name"])
print("Latitude:  ", data["latitude"])
print("Longitude: ", data["longitude"])

Try it with `"check"` which uses the current session's IP address.

In [ ]:
import requests
from google.colab import userdata

api_key = userdata.get("ipstackAPIkey")

# 'check' resolves the IP of the current Colab session
url = "https://api.apilayer.net/ipstack/check"
params = {
    "access_key": api_key
}

response = requests.get(url, params=params)
print(response.json())

In [ ]:
data = response.json()
print("IP address:", data["ip"])
print("City:      ", data["city"])
print("Country:   ", data["country_name"])
print("Latitude:  ", data["latitude"])
print("Longitude: ", data["longitude"])

<b>Why did your IP address not show as a local place?</b>  
<font color=gray>Answer: because your Colab session is running on a virtual machine with a connection to the internet, and an IP address, someplace else.</font>  

In [ ]:
# Compare that use of ipstack's `check` with the `ipify` API
# obtain the public IP address of the current Colab session
my_ip = requests.get("https://api.ipify.org").text
print(f"This session is connected to the internet as {my_ip}.")

The function below wraps the pattern so it can be called with any IP address.
Notice it raises an `Exception` on failure rather than returning silently —
the calling code decides how to handle the error.

In [ ]:
def get_ip_location(ip_address, api_key):
    """Return a geolocation dict for the given IP address using the ipstack API."""
    url = f"https://api.apilayer.net/ipstack/{ip_address}"
    params = {"access_key": api_key}
    response = requests.get(url, params=params)
    if response.status_code != 200:
        raise Exception(f"API request failed: HTTP {response.status_code}")
    data = response.json()
    if data.get("success") is False:
        error = data.get("error", {})
        raise Exception(f"API request failed: {error.get('info', error)}")
    return data

In [ ]:
api_key = userdata.get("ipstackAPIkey")

location = get_ip_location(requests.get("https://api.ipify.org").text, api_key)
print(f"IP address:  {location['ip']}")
print(f"City:        {location['city']}, {location['country_name']}")
print(f"Coordinates: {location['latitude']}, {location['longitude']}")



---



### Example: OMDb — Movie Database API

[omdbapi.com](http://www.omdbapi.com) returns movie metadata from the IMDb database.

**Setup:**
1. Register for a free key at [http://www.omdbapi.com/apikey.aspx](http://www.omdbapi.com/apikey.aspx)
2. Store it in Colab Secrets as `omdbapiAPIkey`

**Key query parameters:**

| Parameter | Description |
|---|---|
| `t` | Exact movie title |
| `s` | Keyword search (returns a list) |
| `y` | Filter by release year |
| `plot` | `short` (default) or `full` |
| `apikey` | Your API key (required) |

In [ ]:
api_key = userdata.get("omdbapiAPIkey")

omdb_url = "http://www.omdbapi.com/"
parameters = {
    "t":      "Inception",
    "plot":   "short",
    "apikey": api_key
}

response = requests.get(omdb_url, params=parameters)
data = response.json()

print("Title:   ", data.get("Title"))
print("Year:    ", data.get("Year"))
print("Director:", data.get("Director"))
print("Rating:  ", data.get("imdbRating"))
print("Plot:    ", data.get("Plot"))

In [ ]:
# Prompt the user for any movie title
movie_title = input("Enter a movie title: ").strip()

parameters = {
    "t":      movie_title,
    "plot":   "short",
    "apikey": api_key
}

response = requests.get(omdb_url, params=parameters)
data = response.json()

if data.get("Response") == "True":
    print(f"\nTitle:    {data.get('Title')}")
    print(f"Year:     {data.get('Year')}")
    print(f"Director: {data.get('Director')}")
    print(f"Plot:     {data.get('Plot')}")
else:
    print(f"Not found: '{movie_title}'")

---

## Example: OpenWeatherMap

[openweathermap.org](https://openweathermap.org) provides current conditions and forecasts
for any location worldwide.

**Setup:**
1. Register for a free key at [https://openweathermap.org](https://openweathermap.org)
2. Store it in Colab Secrets as `openweathermapAPIkey`
3. Manage keys at the [OWM dashboard](https://home.openweathermap.org/api_keys)

The `units` parameter controls temperature scale: `imperial` (°F) or `metric` (°C).

In [ ]:
api_key = userdata.get("openweathermapAPIkey")

owm_url = "http://api.openweathermap.org/data/2.5/weather"
parameters = {
    "q":     "New York, NY, USA",
    "units": "imperial",
    "appid": api_key
}

response = requests.get(owm_url, params=parameters)

if response.status_code == 200:
    data = response.json()
    print("City:        ", data["name"])
    print("Temperature: ", data["main"]["temp"], "°F")
    print("Feels like:  ", data["main"]["feels_like"], "°F")
    print("Humidity:    ", data["main"]["humidity"], "%")
    print("Wind speed:  ", data["wind"]["speed"], "mph")
    print("Description: ", data["weather"][0]["description"])
else:
    print(f"Request failed with status code {response.status_code}.")
    print("Response:", response.json())

The `params` dict approach also works when passing latitude and longitude
instead of a city name — useful for chaining with the ipstack API in Exercise 3.

In [ ]:
# Same request using lat/lon instead of city name
parameters_latlon = {
    "lat":   40.7128,
    "lon":  -74.0060,
    "units": "imperial",
    "appid": api_key
}

response = requests.get(owm_url, params=parameters_latlon)
data = response.json()
print(f"{data['name']}: {data['main']['temp']} °F, {data['weather'][0]['description']}")

---

## Mapping API Data with Folium

`folium` generates interactive maps (backed by Leaflet.js and OpenStreetMap).
A map is created from latitude and longitude coordinates; markers and popups
can be added. The result renders inline in Colab.

In [ ]:
import folium

# Map centered on Baruch College
baruch_map = folium.Map(
    location=[40.74074, -73.98325],
    zoom_start=15,
    control_scale=True,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri'
)

folium.Marker(
    location=[40.74074, -73.98325],
    icon=folium.Icon(icon="book", prefix="fa"),
    popup="Baruch College"
).add_to(baruch_map)

baruch_map

### Chaining APIs: Map a Location from an IP Address

The example below demonstrates **API chaining**: session IP from `ipify`, then coordinates returned by `ipstack`
become the map center passed to `folium`.

In [ ]:
import folium

api_key = userdata.get("ipstackAPIkey")

my_ip = requests.get("https://api.ipify.org").text
location = get_ip_location(my_ip, api_key)
lat = location["latitude"]
lon = location["longitude"]
city = location.get("city", "Unknown")

ip_map = folium.Map(
    location=[lat, lon],
    zoom_start=12,
    control_scale=True,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri'
)

folium.Marker(
    location=[lat, lon],
    icon=folium.Icon(icon="cloud", prefix="fa"),
    popup=f"IP location: {city} ({my_ip})"
).add_to(ip_map)

ip_map

In [ ]:
ip_map.save("my_ip_mapped.html")

---

## Example: polygon.io — Financial Market Data API

[polygon.io](https://polygon.io) provides real-time and historical market data for
stocks, options, forex, and cryptocurrencies. It is one of the primary data sources
used in the financial analytics portion of this course.

**Setup:**
1. Register for a free key at [https://polygon.io](https://polygon.io)
2. Store it in Colab Secrets as `polygonAPIkey`
3. Review the full documentation at [https://polygon.io/docs/](https://polygon.io/docs/)

The endpoint below returns the previous trading day's closing data for any ticker.
Free-tier accounts have a short delay on market data.

In [ ]:
api_key = userdata.get("polygonAPIkey")

ticker = "AAPL"
url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/prev"
parameters = {
    "adjusted": "true",
    "apiKey":   api_key
}

response = requests.get(url, params=parameters)
data = response.json()

print("Status:", data.get("status"))
print("Ticker:", data.get("ticker"))

if data.get("resultsCount", 0) > 0:
    result = data["results"][0]
    print(f"Close:   ${result['c']:.2f}")
    print(f"Open:    ${result['o']:.2f}")
    print(f"High:    ${result['h']:.2f}")
    print(f"Low:     ${result['l']:.2f}")
    print(f"Volume:  {result['v']:,.0f}")

In [ ]:
# Look up any ticker
ticker = input("Enter a stock ticker symbol (e.g. MSFT, TSLA, AMZN): ").strip().upper()

url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/prev"
parameters = {"adjusted": "true", "apiKey": api_key}

response = requests.get(url, params=parameters)
data = response.json()

if data.get("status") == "OK" and data.get("resultsCount", 0) > 0:
    result = data["results"][0]
    print(f"\n{ticker} — Previous Trading Day")
    print(f"  Close:  ${result['c']:.2f}")
    print(f"  Open:   ${result['o']:.2f}")
    print(f"  High:   ${result['h']:.2f}")
    print(f"  Low:    ${result['l']:.2f}")
    print(f"  Volume: {result['v']:,.0f}")
else:
    print(f"No data found for ticker '{ticker}'")

---
## Accessing NYC Open Data With the SODA API

The restaurant inspection dataset used in an earlier module is published through an API. NYC Open Data runs on the Socrata platform, and every dataset on the platform can be queried through the **SODA API** (Socrata Open Data API). The endpoint for the restaurant inspection dataset is:

```
https://data.cityofnewyork.us/resource/43nn-pn8j.json
```

The SODA API accepts query parameters written in **SoQL** (Socrata Query Language). SoQL resembles SQL, and each SoQL parameter name begins with a dollar sign. The parameters are passed to `requests.get()` through the `params` dictionary, exactly as in the REST Countries example.

| SoQL parameter | Purpose | SQL equivalent |
|:---|:---|:---|
| `$select` | Choose columns, or compute aggregates | `SELECT` |
| `$where` | Keep only rows that meet a condition | `WHERE` |
| `$order` | Sort the rows | `ORDER BY` |
| `$group` | Group rows for aggregation | `GROUP BY` |
| `$limit` | Set the maximum number of rows returned (the default is 1,000) | `LIMIT` |

Filtering, sorting, and aggregation in SoQL happen **on the server**. Only the result travels over the network, which is much smaller than the full dataset.

### App Tokens and API Keys

The Developer Settings page of NYC Open Data offers two kinds of credentials: **app tokens** and **API keys**. They serve different purposes and carry different risks.

| | App Token | API Key |
|:---|:---|:---|
| **Purpose** | Identifies the application making requests, so that its requests receive their own request limit instead of sharing the limit for an IP address | Stands in for the account's username and password |
| **What it allows** | Reading public data | Reading, writing, and deleting data, within the limits of the account's role on the portal |
| **Form** | A single value, sent in an `X-App-Token` header | A key ID and secret pair, sent through HTTP Basic authentication |
| **If exposed** | Another person's requests count against the token's request limit | Another person can act on the portal as the account holder |

The restaurant inspection dataset is public, and this notebook only reads it. An app token is therefore sufficient. An API key would add no capability for this work and would carry more risk, because a leaked API key exposes the account itself.

This choice follows the **principle of least privilege**: use the credential with the fewest permissions that still accomplishes the task. Credentials stored with a notebook travel wherever the notebook is shared or submitted, so the least powerful credential limits the damage if one is exposed. An API key becomes necessary only for publishing or changing data, or for reading datasets restricted to signed-in users.

### Obtaining an NYC Open Data App Token

The SODA API identifies applications by an **app token**. Requests without a token share a low request limit with every other request from the same IP address. Colab sessions run on Google servers whose IP addresses are shared by many Colab users, so requests without a token are more likely to be refused with HTTP status `429` (Too Many Requests). Each request that carries a token draws on the token's own limit.

#### Step 1: Create an Account

1. Go to [data.cityofnewyork.us](https://data.cityofnewyork.us) and select **Explore Available Data**.
2. On the catalog page, select **Sign In**.
3. Create an account if you do not have one. The account works on every Socrata-based open data portal, not only on NYC Open Data.

#### Step 2: Create the App Token

1. After signing in, open **Developer Settings** from the account menu. The page is also available directly at [data.cityofnewyork.us/profile/edit/developer_settings](https://data.cityofnewyork.us/profile/edit/developer_settings). The Developer Settings page has two sections, **API Keys** and **App Tokens**. Use the **App Tokens** section. An API key is not needed for this notebook.
2. Select **Create New App Token** and complete the fields:
   * **Name:** The name must be unique across all Socrata-based portals, so include your own name, for example `CST2312 Jane Doe`.
   * **Description:** A short statement of purpose, for example `CST2312 coursework`.
   * **Callback Prefix:** Leave blank.
3. Save the token. The Developer Settings page lists it in a table with the columns **Name**, **Description**, **App Token**, **Secret Token**, and **Actions**.
4. Copy the value in the **App Token** column. The **Secret Token** is not needed for reading data.

Treat the app token as a password. If a token is exposed, delete it on the Developer Settings page and create a replacement.

#### Step 3: Store the Token in Colab Secrets

1. In the left sidebar of Colab, select the key icon (**Secrets**).
2. Select **Add new secret**.
3. In the **Name** field, enter `nycAppToken`. The name must match exactly, including capitalization.
4. In the **Value** field, paste the app token.
5. Turn on the **Notebook access** toggle for this notebook.

#### Step 4: Retrieve the Token in Code

The SODA API receives the token in an HTTP header named `X-App-Token`. The cell below reads the token from Colab Secrets and builds the header dictionary used by every request in this section.

In [ ]:
import requests
import pandas as pd
from google.colab import userdata

nyc_app_token = userdata.get("nycAppToken")
nyc_headers = {"X-App-Token": nyc_app_token}

inspections_url = "https://data.cityofnewyork.us/resource/43nn-pn8j.json"

### A First Request: `$limit`

The request below asks for five records. The `response.url` attribute shows the URL that `requests` built from the `params` dictionary. The dollar sign in `$limit` appears in the URL as `%24`, its encoded form.

In [ ]:
params = {"$limit": 5}

response = requests.get(inspections_url, params=params, headers=nyc_headers)

print("Status code:", response.status_code)
print("Request URL:", response.url)

if response.status_code == 200:
    records = response.json()        # a list of dictionaries, one per record
    print("Records returned:", len(records))
else:
    print(response.text[:300])

The response is a list of dictionaries in the same form as the JSON file read in an earlier module. Passing the list to `pd.DataFrame()` produces a DataFrame with one row per record and one column per key. As before, every value arrives as a string.

In [ ]:
sample = pd.DataFrame(records)
sample.head()

**Troubleshooting**

| Status code | Meaning | Action |
|:---|:---|:---|
| `200` | The request succeeded. | None. |
| `400` | The SoQL in `params` could not be parsed. | Check column names, quotation marks, and spelling of the SoQL keywords. |
| `403` | The app token was rejected. | Check the value stored in Colab Secrets against the **App Token** column on the Developer Settings page. |
| `429` | Too many requests in a short period. | Wait before sending the request again, and confirm that the request carries the `X-App-Token` header. |

### Choosing Columns and Rows: `$select` and `$where`

`$select` lists the columns to return, separated by commas. `$where` states the condition each returned row must meet.

**Quotation marks.** In SoQL, a text value is enclosed in **single quotes**, as in `zipcode='11201'`. The SoQL condition is itself a Python string, so the Python string uses double quotes on the outside and SoQL single quotes on the inside:

```python
"$where": "zipcode='11201'"
```

Reversing the quotation marks, or omitting the single quotes around `11201`, produces an HTTP `400` error.

In [ ]:
params = {
    "$select": "camis, dba, zipcode, grade, score, inspection_date",
    "$where":  "zipcode='11201'",
    "$limit":  50000,
}

response = requests.get(inspections_url, params=params, headers=nyc_headers)
print("Status code:", response.status_code)

zip_11201 = pd.DataFrame(response.json())
print(f"Records for ZIP 11201: {len(zip_11201):,}")
zip_11201.head()

Conditions are combined with `AND` and `OR`. The keyword `IS NOT NULL` keeps only rows in which a column has a value:

```python
"$where": "zipcode='11201' AND grade IS NOT NULL"
```

Date columns can be compared with a date written in single quotes:

```python
"$where": "zipcode='11201' AND inspection_date >= '2025-01-01'"
```

### Sorting: `$order`

`$order` names the column to sort by, followed by `ASC` (ascending, the default) or `DESC` (descending). Without `$order`, the API does not guarantee the order of the rows it returns. The request below returns the ten most recent inspections in ZIP 11201.

In [ ]:
params = {
    "$select": "dba, grade, score, inspection_date",
    "$where":  "zipcode='11201'",
    "$order":  "inspection_date DESC",
    "$limit":  10,
}

response = requests.get(inspections_url, params=params, headers=nyc_headers)
recent_11201 = pd.DataFrame(response.json())
recent_11201

### Aggregating on the Server: `$select` With `$group`

SoQL computes aggregates such as `count(*)`, `avg()`, `min()`, and `max()` on the server. The `AS` keyword names the result column. Every column in `$select` that is not an aggregate must also appear in `$group`.

The request below counts inspection records in each borough. The server returns six rows, one per borough value, instead of every inspection record.

In [ ]:
params = {
    "$select": "boro, count(*) AS inspection_count",
    "$group":  "boro",
    "$order":  "inspection_count DESC",
}

response = requests.get(inspections_url, params=params, headers=nyc_headers)
borough_counts = pd.DataFrame(response.json())
borough_counts

The counts arrive as strings, like every other value from the SODA API. They must be converted with `pd.to_numeric()` before any arithmetic.

In [ ]:
borough_counts["inspection_count"] = pd.to_numeric(borough_counts["inspection_count"])
print("Total inspection records:", f"{borough_counts['inspection_count'].sum():,}")
borough_counts.dtypes

---
## Exercise: ZIP Code Grade Report From the API

A community board has asked for a report on restaurant grades in a single ZIP code, produced directly from the NYC Open Data API. The report compares a count computed **on the server** with the same count computed **in pandas**.

This exercise has four stages. Each stage has:

* A short description of what the stage produces.
* A **skeleton** code cell with a `TODO` comment showing where to write your code.
* A **verification** cell with `assert` statements that check the structure of your result.
* A collapsible **solution** block. Try each stage on your own first; expand the solution only after a serious attempt.

The recommended ZIP is **11201** (Downtown Brooklyn near City Tech). Any New York City ZIP code may be substituted.

### Stage 1: Build the Query Parameters

Create a dictionary named `grade_params` that requests, for the ZIP code in `target_zip`:

* the columns `camis`, `dba`, `zipcode`, and `grade`;
* only rows whose `zipcode` equals `target_zip` **and** whose `grade` is not null;
* up to 50,000 rows.

Build the `$where` condition with an f-string so that it uses the value of `target_zip`.

In [ ]:
target_zip = "11201"

# TODO: build the query parameter dictionary described above.
# Hint: the $where value must place target_zip inside SoQL single quotes.
grade_params = ...

grade_params

In [ ]:
# Stage 1 verification
assert isinstance(grade_params, dict), "grade_params must be a dictionary"
for key in ["$select", "$where", "$limit"]:
    assert key in grade_params, f"grade_params is missing the {key} key"
assert f"'{target_zip}'" in grade_params["$where"], \
    "the $where value must contain target_zip inside single quotes"
assert "grade IS NOT NULL" in grade_params["$where"], \
    "the $where value must exclude rows with no grade"
for column in ["camis", "dba", "zipcode", "grade"]:
    assert column in grade_params["$select"], f"$select must include {column}"
assert int(grade_params["$limit"]) >= 50000, "$limit must be at least 50000"
print("OK: Stage 1 passed.")

<details>
<summary><b>Solution: Stage 1</b></summary>

```python
grade_params = {
    "$select": "camis, dba, zipcode, grade",
    "$where":  f"zipcode='{target_zip}' AND grade IS NOT NULL",
    "$limit":  50000,
}
```

The f-string inserts the value of `target_zip` between the SoQL single quotes. For the ZIP `"11201"`, the resulting condition is `zipcode='11201' AND grade IS NOT NULL`.

</details>

### Stage 2: Send the Request and Build a DataFrame

Send a GET request to `inspections_url` with `grade_params` and the `nyc_headers` dictionary. If the status code is `200`, convert the JSON response to a DataFrame named `graded_inspections`.

In [ ]:
# TODO: send the request with grade_params and nyc_headers.
response = ...

# TODO: convert the JSON response to a DataFrame named graded_inspections.
graded_inspections = ...

print("Status code:", response.status_code)
print(f"Graded inspection records: {len(graded_inspections):,}")

In [ ]:
# Stage 2 verification
assert response.status_code == 200, f"request failed with status {response.status_code}: {response.text[:200]}"
assert isinstance(graded_inspections, pd.DataFrame), "graded_inspections must be a DataFrame"
assert len(graded_inspections) > 0, "no records returned: check target_zip"
assert (graded_inspections["zipcode"] == target_zip).all(), "every zipcode must equal target_zip"
assert graded_inspections["grade"].notna().all(), "no grade value may be missing"
print(f"OK: Stage 2 passed with {len(graded_inspections):,} records.")

<details>
<summary><b>Solution: Stage 2</b></summary>

```python
response = requests.get(inspections_url, params=grade_params, headers=nyc_headers)
graded_inspections = pd.DataFrame(response.json())
```

Because the server applied the `$where` condition, every row in the DataFrame already belongs to `target_zip` and carries a grade. No filtering in pandas is needed.

</details>

### Stage 3: Count Grades on the Server

Write a second request that asks the server to count graded inspection records by grade for `target_zip`. Name the count column `grade_count`, sort the result by `grade`, and store the result in a DataFrame named `server_counts`. Convert `grade_count` to a number with `pd.to_numeric()`.

In [ ]:
# TODO: build the parameters for a server-side count of records by grade.
count_params = ...

# TODO: send the request and convert the response to a DataFrame named server_counts.
response = ...
server_counts = ...

# TODO: convert the grade_count column to a number.


server_counts

In [ ]:
# Stage 3 verification
assert isinstance(server_counts, pd.DataFrame), "server_counts must be a DataFrame"
assert set(server_counts.columns) == {"grade", "grade_count"}, \
    "server_counts must have exactly the columns grade and grade_count"
assert pd.api.types.is_numeric_dtype(server_counts["grade_count"]), \
    "grade_count must be converted to a number"
assert server_counts["grade"].is_unique, "each grade must appear only once"
assert list(server_counts["grade"]) == sorted(server_counts["grade"]), "rows must be sorted by grade"
print("OK: Stage 3 passed.")

<details>
<summary><b>Solution: Stage 3</b></summary>

```python
count_params = {
    "$select": "grade, count(*) AS grade_count",
    "$where":  f"zipcode='{target_zip}' AND grade IS NOT NULL",
    "$group":  "grade",
    "$order":  "grade",
}

response = requests.get(inspections_url, params=count_params, headers=nyc_headers)
server_counts = pd.DataFrame(response.json())
server_counts["grade_count"] = pd.to_numeric(server_counts["grade_count"])
```

The `$where` condition is the same as in Stage 1. The `$group` parameter makes the server return one row per grade instead of one row per inspection record.

</details>

### Stage 4: Confirm the Server Count in Pandas

Count the grades in `graded_inspections` with pandas and store the result in a DataFrame named `pandas_counts` with the same two columns as `server_counts`, sorted by `grade`. Then compare the two DataFrames. The two counts come from the same records, so they must agree.

In [ ]:
# TODO: count grades in graded_inspections with pandas.
# Hint: .groupby("grade").size() returns a Series; .reset_index(name="grade_count")
# turns it into a DataFrame with the columns grade and grade_count.
pandas_counts = ...

pandas_counts

In [ ]:
# Stage 4 verification
assert isinstance(pandas_counts, pd.DataFrame), "pandas_counts must be a DataFrame"
assert set(pandas_counts.columns) == {"grade", "grade_count"}, \
    "pandas_counts must have exactly the columns grade and grade_count"
comparison = pd.merge(server_counts, pandas_counts, on="grade",
                      how="outer", suffixes=("_server", "_pandas"))
assert (comparison["grade_count_server"] == comparison["grade_count_pandas"]).all(), \
    "the server and pandas counts do not agree"
print("OK: Stage 4 passed. The server and pandas counts agree.")
comparison

<details>
<summary><b>Solution: Stage 4</b></summary>

```python
pandas_counts = (
    graded_inspections
    .groupby("grade")
    .size()
    .reset_index(name="grade_count")
    .sort_values("grade")
    .reset_index(drop=True)
)
```

The verification cell merges the two DataFrames on `grade` with an outer join, so a grade that appears in only one of the two results would produce a `NaN` count and fail the comparison.

The two approaches differ in what travels over the network. Stage 2 transferred every graded inspection record so that pandas could count them. Stage 3 transferred only one row per grade. For large datasets, aggregating on the server is often the only practical approach.

</details>



---



## Business Analytics & Other APIs

This curated list focuses on free or freemium APIs suitable for:
- Business analytics  
- Data science  
  
Each entry includes a description and direct URL.
  
Choose one or more of the following APIs, read the documentation and practice using requests to the API.
---

### 1. General Data & Economic Indicators

#### Data USA
- Description: U.S. demographic, economic, and workforce data
- Use Cases: Data visualization, regression analysis, regional comparisons
- URL: https://datausa.io/about/api/

#### World Bank API
- Description: Global economic indicators (GDP, inflation, population, etc.)
- Use Cases: Time series analysis, macroeconomic modeling
- URL: https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation

#### OECD API
- Description: International economic and social data
- Use Cases: Comparative country analysis, policy evaluation
- URL: https://data.oecd.org/api/

---

### 2. Financial & Market Data

#### Alpha Vantage
- Description: Stock prices, technical indicators, FX data
- Use Cases: Financial modeling, forecasting, portfolio analysis
- URL: https://www.alphavantage.co/documentation/

#### CoinGecko API
- Description: Cryptocurrency prices, volumes, market data
- Use Cases: Volatility analysis, alternative asset modeling
- URL: https://www.coingecko.com/en/api/documentation

#### Financial Modeling Prep
- Description: Company financials, income statements, ratios
- Use Cases: Fundamental analysis, valuation exercises
- URL: https://site.financialmodelingprep.com/developer/docs/

---

### 3. Business & Company Data

#### Clearbit Logo API
- Description: Company logos and metadata
- Use Cases: Dashboard design, entity enrichment
- URL: https://clearbit.com/docs#logo-api

#### OpenCorporates API
- Description: Global company registry data
- Use Cases: Entity resolution, corporate network analysis
- URL: https://api.opencorporates.com/documentation/API-Reference

---

### 4. Marketing & Customer Analytics

#### HubSpot API (Free Tier)
- Description: CRM data (contacts, deals, pipelines)
- Use Cases: Customer segmentation, sales funnel analysis
- URL: https://developers.hubspot.com/docs/api/overview

#### Mailchimp API
- Description: Email campaign and audience data
- Use Cases: Marketing analytics, A/B testing scenarios
- URL: https://mailchimp.com/developer/marketing/api/

---

### 5. Social Media & Sentiment Data

#### Reddit API
- Description: Posts, comments, and engagement data
- Use Cases: Sentiment analysis, trend detection
- URL: https://www.reddit.com/dev/api/

#### Twitter/X API (Free Tier - limited)
- Description: Tweets and engagement metrics
- Use Cases: Social listening, text analytics
- URL: https://developer.twitter.com/en/docs/twitter-api

---

### 6. Operational & Logistics Data

#### Aviationstack
- Description: Flight and airline data
- Use Cases: Operations analytics, scheduling optimization
- URL: https://aviationstack.com/documentation

#### OpenWeather API
- Description: Weather data globally
- Use Cases: Demand forecasting, supply chain scenarios
- URL: https://openweathermap.org/api

---

### 7. Teaching / Prototyping APIs (Low Friction)

#### JSONPlaceholder
- Description: Fake REST API for testing CRUD operations
- Use Cases: API integration exercises, ETL pipelines
- URL: https://jsonplaceholder.typicode.com/

#### REST Countries
- Description: Country-level demographic and geographic data
- Use Cases: Data joins, enrichment tasks
- URL: https://restcountries.com/

#### Numbers API
- Description: Numeric trivia (dates, math, facts)
- Use Cases: Lightweight API demonstrations
- URL: http://numbersapi.com/

---

### 8. API Discovery Resources (Filtered for Analytics Use)

#### Public APIs (GitHub)
- Description: Master list of APIs; filter by “Finance”, “Data”, “Business”
- URL: https://github.com/public-apis/public-apis

#### APIs.guru
- Description: Structured OpenAPI directory
- Use Cases: Schema-driven API exploration
- URL: https://apis.guru/

---

## Exercises

### Exercise 1 — No API Key Required

Using the `restcountries.com` API (no key required), write code that:

a. Prompts the user for a country name  
b. Retrieves the country data  
c. Prints: country name, capital, population, region, and the list of official languages  
d. Handles the case where the country name is not found (status 404)  

*Hint: languages are stored under `country["languages"]` as a dictionary mapping  
language codes to names, e.g., `{"eng": "English", "fra": "French"}`.*

In [ ]:
# TODO — Exercise 1

### Solution for Exercise 1

In [ ]:
# ══ SOLUTION ══
# solution provided after assignments are submitted

### Exercise 2 — API Key and Query Parameters

Using the OpenWeatherMap API:

a. Query current weather for New York, NY in **metric** units  
b. Extract and print: city name, temperature (°C), humidity (%), wind speed (m/s),
   and weather description  
c. Modify the request to use **latitude and longitude** instead of a city name  
   *(Replace the `q` parameter with `lat` and `lon`.
   See the [OpenWeatherMap docs](http://openweathermap.org/current#geo).)*  
d. Retrieve the weather for **San Francisco, CA** using the city name parameter

In [ ]:
# TODO — Exercise 2a and 2b

In [ ]:
# TODO — Exercise 2c  (lat/lon)

In [ ]:
# TODO — Exercise 2d  (San Francisco)

### Solution for Exercise 2

In [ ]:
# ══ SOLUTION ══
# solution provided after assignments are submitted

### Exercise 3 — Chaining Two APIs

Write a single block of code that:

1. Uses the ipstack API to look up the geographic location of IP address `8.8.8.8`
2. Extracts the latitude and longitude from the ipstack response
3. Uses those coordinates (with `lat` / `lon` parameters) to query OpenWeatherMap
   for the current weather at that location
4. Prints a summary: IP address, city, country, temperature (°F), and weather description

This exercise demonstrates a common real-world pattern: the output of one API call
becomes the input of the next.

In [ ]:
# TODO — Exercise 3

### Solution for Exercise 3

In [ ]:
# ══ SOLUTION ══
# solution provided after assignments are submitted

---

> *Submit your solutions on Brightspace. Download your completed notebook as an `.ipynb` file
> (File → Download → Download .ipynb) and upload it to the Module  assignment.
> The Brightspace assignment page has full submission details.*

---

## Key Terms

| Term | Definition |
|---|---|
| **API** | A documented contract defining how one program requests services from another |
| **Web API** | An API accessible over the internet using HTTP |
| **REST** | Representational State Transfer — an architectural style for web APIs using HTTP methods and resource-based URLs |
| **Endpoint** | A specific URL at which an API service is accessible |
| **HTTP GET** | The HTTP method used to retrieve data without modifying server state |
| **Status code** | An integer in the HTTP response: 200 = success; 4xx = client error; 5xx = server error |
| **JSON** | JavaScript Object Notation — a text format for structured data that maps to Python dicts and lists |
| **API key** | A credential string passed with requests to authenticate the caller and enforce rate limits |
| **Rate limiting** | A restriction on the number of API requests allowed per unit of time |
| **Query parameter** | A key-value pair appended to a URL (after `?`) that passes arguments to an endpoint |
| **`requests` library** | A Python library for sending HTTP requests and parsing responses |
| **`response.json()`** | Parses the JSON response body and returns a Python dictionary or list |
| **Geolocation** | Determining a physical location (lat/lon) from digital data such as an IP address |
| **API chaining** | Using the output of one API call as the input to a second API call |
| **SOA** | Service-Oriented Architecture — a design pattern in which applications are composed of loosely coupled services communicating over a network |



---

